In [1]:
# YouTube Engagement Metrics Collector
# K-pop Sentiment Analysis Project
# Collects views, likes, and comment count for each comeback MV

from googleapiclient.discovery import build
import pandas as pd
from datetime import datetime

# --- Read API key from credentials file ---
with open("../credentials/api_keys.txt", "r") as f:
    for line in f:
        if line.startswith("YOUTUBE_API_KEY"):
            API_KEY = line.strip().split("=")[1]

# --- Video IDs ---
VIDEOS = {
    "aespa_Whiplash": "jWQx2f-CErU",
    "IVE_RebelHeart": "g36q0ZLvygQ",
    "TWICE_Strategy": "Sz_wWzgh-vQ",
    "NCTDREAM_WhenImWithYou": "B1qq8IvzSz4",
    "SEVENTEEN_Thunder": "pS57UX6s-xw",
    "StrayKids_ChkChkBoom": "0P0aQreFs8w"
}

# --- Build YouTube client ---
youtube = build("youtube", "v3", developerKey=API_KEY)

# --- Collect metrics ---
results = []

for name, video_id in VIDEOS.items():
    request = youtube.videos().list(
        part="statistics,snippet",
        id=video_id
    )
    response = request.execute()
    
    if response["items"]:
        item = response["items"][0]
        stats = item["statistics"]
        snippet = item["snippet"]
        
        results.append({
            "group_comeback": name,
            "video_id": video_id,
            "title": snippet["title"],
            "published_at": snippet["publishedAt"],
            "view_count": int(stats.get("viewCount", 0)),
            "like_count": int(stats.get("likeCount", 0)),
            "comment_count": int(stats.get("commentCount", 0)),
            "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })
        print(f"Collected: {name}")

# --- Save to CSV ---
df = pd.DataFrame(results)
print(df)
df.to_csv("../07_tableau/youtube_metrics_snapshot.csv", index=False)
print("\nSaved successfully.")

Collected: aespa_Whiplash
Collected: IVE_RebelHeart
Collected: TWICE_Strategy
Collected: NCTDREAM_WhenImWithYou
Collected: SEVENTEEN_Thunder
Collected: StrayKids_ChkChkBoom
           group_comeback     video_id  \
0          aespa_Whiplash  jWQx2f-CErU   
1          IVE_RebelHeart  g36q0ZLvygQ   
2          TWICE_Strategy  Sz_wWzgh-vQ   
3  NCTDREAM_WhenImWithYou  B1qq8IvzSz4   
4       SEVENTEEN_Thunder  pS57UX6s-xw   
5    StrayKids_ChkChkBoom  0P0aQreFs8w   

                                              title          published_at  \
0                           aespa 에스파 'Whiplash' MV  2024-10-21T09:00:42Z   
1                          IVE 아이브 'REBEL HEART' MV  2025-01-13T09:01:36Z   
2  TWICE “Strategy (feat. Megan Thee Stallion)” M/V  2024-12-06T04:59:07Z   
3           NCT DREAM 엔시티 드림 'When I’m With You' MV  2024-11-11T09:00:09Z   
4             SEVENTEEN (세븐틴) 'THUNDER' Official MV  2025-05-26T08:55:08Z   
5                     Stray Kids "Chk Chk Boom" M/V  2024-07-19T04:00:

In [2]:
# YouTube Comments Collector
# Collects up to 500 comments per comeback video for sentiment analysis

import time

def get_comments(youtube, video_id, max_comments=500):
    comments = []
    next_page_token = None
    
    while len(comments) < max_comments:
        request = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=100,
            pageToken=next_page_token,
            textFormat="plainText"
        )
        response = request.execute()
        
        for item in response["items"]:
            comment = item["snippet"]["topLevelComment"]["snippet"]
            comments.append({
                "video_id": video_id,
                "comment": comment["textDisplay"],
                "likes": comment["likeCount"],
                "published_at": comment["publishedAt"]
            })
        
        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break
            
        time.sleep(0.5)
    
    return comments[:max_comments]

# --- Collect comments for all videos ---
all_comments = []

for name, video_id in VIDEOS.items():
    print(f"Collecting comments for {name}...")
    comments = get_comments(youtube, video_id, max_comments=500)
    
    for c in comments:
        c["group_comeback"] = name
    
    all_comments.extend(comments)
    print(f"  Collected {len(comments)} comments")
    time.sleep(1)

# --- Save to CSV ---
comments_df = pd.DataFrame(all_comments)
print(f"\nTotal comments collected: {len(comments_df)}")
print(comments_df.head())
comments_df.to_csv("../01_raw_data/youtube/youtube_comments_raw.csv", index=False)
print("\nSaved to 01_raw_data/youtube/youtube_comments_raw.csv")

  Collected 500 comments
  Collected 500 comments
  Collected 500 comments
  Collected 500 comments
  Collected 500 comments
  Collected 500 comments

Total comments collected: 3000
      video_id                                            comment  likes  \
0  jWQx2f-CErU                                             Splash      0   
1  jWQx2f-CErU  저가문제인들인적아는대로만적었어요참고하세요여기명단문제인이저희집수급비찝적대지못하게여러분...      0   
2  jWQx2f-CErU  장희원노은아정지원황제희전해령저랑중1같은반박민희는중2때또김지은,이금비(같은반),박은지...      0   
3  jWQx2f-CErU  2지켜주세요이거내용돌아가면서말좀전해주세요저그리고넥플이랑유튜브군대군대일부러댓글적는거에...      0   
4  jWQx2f-CErU  1저가외국여행가는거랑바다에서튜브탄다는거랑저가직접스쿠터주행그런거위험이유로이금비장희원박...      0   

           published_at  group_comeback  
0  2026-03-13T15:47:28Z  aespa_Whiplash  
1  2026-03-13T14:09:12Z  aespa_Whiplash  
2  2026-03-13T14:08:58Z  aespa_Whiplash  
3  2026-03-13T14:08:47Z  aespa_Whiplash  
4  2026-03-13T14:08:37Z  aespa_Whiplash  

Saved to 01_raw_data/youtube/youtube_comments_raw.csv
